# Training: 30-Class EfficientNet-B0

Fine-tune an ImageNet-pretrained EfficientNet-B0 for 30-class fruit classification (Fruits-360 subset).

In [8]:
import timm
import torch
import torch.nn as nn

NUM_CLASSES = 30
model = timm.create_model('efficientnet_b0', pretrained=False, num_classes=NUM_CLASSES)

print(f"Model: {model.default_cfg['architecture']}")
print(f"Classifier head: {model.get_classifier()}")

Model: efficientnet_b0
Classifier head: Linear(in_features=1280, out_features=30, bias=True)


In [ ]:
from torchvision import datasets
from torch.utils.data import DataLoader
import os

config = timm.data.resolve_model_data_config(model)
train_transform = timm.data.create_transform(**config, is_training=True)
val_transform = timm.data.create_transform(**config, is_training=False)

train_ds = datasets.ImageFolder("PrepData/Training", transform=train_transform)
val_ds = datasets.ImageFolder("PrepData/Validation", transform=val_transform)

num_workers = min(8, os.cpu_count() or 1)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=num_workers, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=num_workers, pin_memory=True)

print(f"Classes: {train_ds.classes}")
print(f"Training samples: {len(train_ds)}, Validation samples: {len(val_ds)}")
print(f"DataLoader workers: {num_workers}")

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
print(f"Device: {device}")

Device: cuda


In [ ]:
from tqdm.auto import tqdm

epochs = 15
for epoch in range(epochs):
    # Training Phase
    model.train()
    train_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # Validation Phase
    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / len(val_ds)
    print(f"Epoch {epoch+1}: Loss = {train_loss/len(train_loader):.4f}, Val Acc = {accuracy:.2f}%")

Epoch 1/15:   0%|          | 0/315 [00:00<?, ?it/s]

Epoch 1: Loss = 2.4032, Val Acc = 68.72%


Epoch 2/15:   0%|          | 0/315 [00:00<?, ?it/s]

In [ ]:
torch.save(model.state_dict(), "fruit_classifier_30cls_b0.pth")